# dscim-cli walkthrough

dscim is the Climate Impact Lab's library for computing the social cost
of carbon (SCC): it turns sector-level climate damages into damage
functions, applies them to FaIR climate projections, discounts, and
integrates to an SCC. dscim-cli drives that library from a YAML config.

This notebook runs entirely on synthetic fixtures generated below. No
external data is needed, but the `run` extra must be installed
(`uv pip install ".[run]"`) so dscim itself is available.

## Generate fixtures and a config

The test suite's fixture factory writes tiny versions of every input a
run needs: an economics zarr, reduced-damage zarrs, a GMST table, FaIR
temperature projections, and a pulse conversion file. We point a config
at them and save it as `demo_data/demo.yml`.

In [ ]:
import pathlib
import sys

import yaml

sys.path.insert(0, str(pathlib.Path("..") / "tests"))
import fixture_factory

data = pathlib.Path("demo_data")
data.mkdir(exist_ok=True)
config = fixture_factory.ssp_fixture_config(data)
config_path = data / "demo.yml"
config_path.write_text(yaml.safe_dump(config))
print(f"wrote {config_path}")

## What is the pipeline?

`stages` explains the pipeline without touching any data: each stage,
what it reads and writes, and which dimensions it collapses. dscim's
central moves are all dimension collapses, so this is the map to keep
in mind.

In [ ]:
!dscim-cli stages

## What can be configured?

`options` lists every option dscim accepts. Each carries a status:
`supported`, `unsupported` (accepted by dscim but broken or restricted,
with the reason), `dead` (accepted and ignored), or `removed` (only on
other dscim branches). Here are the unsupported ones:

In [ ]:
!dscim-cli options --status unsupported

`explain` gives the full record for one option, including why a
value is unsupported and where in dscim's source that behavior lives:

In [ ]:
!dscim-cli explain discounting_type constant_gwr

## Which combinations are valid?

Some restrictions span options. `constraints` lists them with their
source citations; the same data drives validation, so what it prints is
what `validate` enforces.

In [ ]:
!dscim-cli constraints

## Check the config

`validate` reports every problem at once. Our generated config is
valid:

In [ ]:
!dscim-cli validate demo_data/demo.yml

## What would run, in what order?

`plan` lays out the pipeline for this config as ordered steps, each
marked ready or blocked, with the command that produces every missing
input. Our fixtures already include reduced damages, so the run step is
ready:

In [ ]:
!dscim-cli plan demo_data/demo.yml

A dry run first prints the settings the sweep will use, marking
which values came from the config and which are dscim defaults (the
result-selecting values must always be explicit), then summarizes the
expanded runs:

In [ ]:
!dscim-cli run demo_data/demo.yml --dry-run

## Run it

One real run: fit the damage function on the fixture damages, apply it
to the FaIR projections, discount, and write SCC files. On these tiny
fixtures this takes a few seconds.

In [ ]:
!dscim-cli run demo_data/demo.yml

## Look at the result

Every run writes its artifacts plus a `*_run_metadata.yaml` recording
the resolved settings with their provenance, the dscim version and
commit that produced the result, and dependency versions.

In [ ]:
import xarray as xr

results = data / "results" / "labor" / "2020" / "unmasked"
stem = "adding_up_euler_ramsey_eta2.0_rho0.0001"
scc = xr.open_dataset(results / f"{stem}_scc.nc4")
print(scc)

In [ ]:
metadata = yaml.safe_load((results / f"{stem}_run_metadata.yaml").read_text())
print("dscim:", metadata["dscim_version"], "commit", metadata["dscim_commit"])
print("eta came from:", metadata["provenance"]["eta"])
print("ext_method came from:", metadata["provenance"]["ext_method"])

## Where to go next

Real inputs replace the fixtures: `examples/ssp.yaml` shows the full
discrete-SSP surface (aggregate sectors, ECS masks, the combine step)
and `examples/rff.yaml` the EPA/RFF mode with precomputed damage
functions and the `scc` composition step. `dscim-cli defaults CONFIG`
shows every effective value for a config and where it came from.